# DukaStock — Notebook 2: ML Benchmark Experiment

**Primary research question:** Across SARIMA, Prophet, XGBoost, and N-BEATS,
at what minimum data density does each model class first achieve statistically
significant improvement over a naive last-week-sales baseline (p < 0.05,
Diebold-Mariano test with Newey-West variance correction), when trained under
simulated cold-start conditions at 6 density levels (5 / 15 / 30 / 50 / 75 /
100 %) using walk-forward cross-validation with a 7-day horizon, **evaluated
at the individual store (single-Duka proxy) level**?

### Why store-level evaluation, not national aggregate

DukaStock serves individual shopkeepers, not national retailers. Aggregating
10 stores into one national series inflates lag-7 autocorrelation to r = 0.94
(R² = 88 %), leaving only 12 % of variance for any model to improve on and
making Diebold-Mariano significance nearly unachievable regardless of model
quality. At the single-store level, lag-7 autocorrelation drops to r = 0.55–
0.80 (R² = 30–64 %) depending on product, giving a realistic margin for ML
improvement and a fair DM test. Each Kaggle 'store' serves as a proxy for
one Duka; metrics are computed per store and then averaged.

### Honest limitation on Rwanda localisation

The underlying dataset is the **Kaggle Store Item Demand Forecasting
Challenge** (anonymous retail data, origin unknown). The Rwanda localisation
layer — holiday calendar, Genocide Memorial Day suppressor, rainy-season
intensity, product naming — is **fully implemented and designed for
deployment** on real Duka sales data. It cannot be validated on this proxy
dataset because the sales values have no relationship to Rwandan public
events. This is disclosed explicitly here and in the thesis methods section.

### Running on Google Colab (recommended)

1. Upload this notebook to [colab.research.google.com](https://colab.research.google.com).
2. Enable GPU: **Runtime → Change runtime type → T4 GPU**.
3. Run **Cell 0** first — it mounts your Google Drive at `/content/drive/MyDrive/DukaStock/`.
4. Upload `fmcg_rwanda_localized.csv` into that Drive folder before running Cell 3.
5. After Cell 12 finishes, results are automatically saved back to Drive.

**Expected runtime:** ~10–14 hours (Colab/Kaggle GPU). Set
`RUN_ALL_STORES = False` in Cell 11 to use 5 stores (~5–7 hours) if the
session limit is a concern.


In [ ]:
# Cell 0 — Environment detection: Google Colab, GPU, and Google Drive
import os, sys

!pip install torch

# ── Detect whether we are inside Google Colab ────────────────────────────────
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    # google.colab can be importable in non-Colab environments too (e.g.
    # some Kaggle images) -- the actual Colab VM is confirmed by this marker
    # file, the same one google.colab.drive checks internally before
    # attempting to mount. Without this check, drive.mount() below crashes
    # with NotImplementedError on any environment where the package merely
    # happens to be installed but isn't really Colab.
    IN_COLAB = os.path.exists('/var/colab/hostname')
except ImportError:
    pass

if IN_COLAB:
    print("Running on Google Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    # All DukaStock files (CSV input, results output) go here.
    DRIVE_ROOT = "/content/drive/MyDrive/DukaStock"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"Google Drive mounted. DukaStock folder: {DRIVE_ROOT}")
    print("  → Place fmcg_rwanda_localized.csv inside that Drive folder before running Cell 3.")
else:
    DRIVE_ROOT = None
    print("Not running on Google Colab — using local paths only.")

# ── GPU check (logged here; used in Cell 10 for N-BEATS accelerator) ─────────
import torch
ON_GPU = torch.cuda.is_available()
print("CUDA available:", ON_GPU)
if ON_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU memory: {total_mem / 1e9:.1f} GB")
else:
    print("No GPU — N-BEATS will run on CPU (slower). SARIMA/Prophet/XGBoost are unaffected.")
    print("  On Colab:  Runtime → Change runtime type → T4 GPU")
    print("  On Kaggle: Settings → Accelerator → GPU T4 x2")


In [ ]:
# Cell 1 — Install dependencies
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# accelerate: required by neuralforecast (N-BEATS) for GPU training
required = [
    ("pmdarima",       "pmdarima"),
    ("prophet",        "prophet"),
    ("neuralforecast", "neuralforecast"),
    ("seaborn",        "seaborn"),
    ("accelerate",     "accelerate"),
]
for pkg, import_name in required:
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {pkg} ...")
        pip_install([pkg])

print("Dependencies ready.")


In [ ]:
# Cell 2 — Imports and reproducibility seed
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Fix all random seeds for full reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

try:
    import pmdarima as pm
    _SARIMA_OK = True
except ImportError:
    _SARIMA_OK = False

try:
    from prophet import Prophet
    _PROPHET_OK = True
except ImportError:
    _PROPHET_OK = False

try:
    import xgboost as xgb
    from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
    _XGB_OK = True
except ImportError:
    _XGB_OK = False

try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NBEATS
    _NBEATS_OK = True
except ImportError:
    _NBEATS_OK = False

print(f"SARIMA available: {_SARIMA_OK} | Prophet available: {_PROPHET_OK} | "
      f"XGBoost available: {_XGB_OK} | N-BEATS available: {_NBEATS_OK}")
print(f"Reproducibility seed: {RANDOM_SEED}")


In [ ]:
# Cell 3 — Load the Rwanda-localized dataset produced by Notebook 1
#
# We keep the data at the individual-store level. Each (store, product)
# pair is a proxy for one Duka. The experiment loop in Cell 11 iterates
# over all (store, product) pairs and averages metrics across stores.

import glob

CANDIDATE_PATHS = [
    Path(f"{DRIVE_ROOT}/fmcg_rwanda_localized.csv") if DRIVE_ROOT else None,
    Path("/kaggle/working/fmcg_rwanda_localized.csv"),
    # recursive, not just one level deep -- covers datasets where the CSV
    # ended up nested inside a subfolder rather than at the dataset root
    *[Path(p) for p in glob.glob("/kaggle/input/**/fmcg_rwanda_localized.csv", recursive=True)],
    Path("../data/fmcg_rwanda_localized.csv"),
    Path("ml_experiments/data/fmcg_rwanda_localized.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p and p.exists()), None)
if DATA_PATH is None:
    # Show what actually IS there, so a missing/misnamed dataset attachment
    # is obvious immediately instead of requiring a separate debug cell.
    kaggle_input_listing = "(none)"
    if Path("/kaggle/input").exists():
        found = list(Path("/kaggle/input").rglob("*"))
        kaggle_input_listing = "\n".join(f"    {p}" for p in found[:50]) or "(empty)"
    raise FileNotFoundError(
        "fmcg_rwanda_localized.csv not found.\n"
        "  On Colab:  upload it to MyDrive/DukaStock/ and re-run this cell.\n"
        "  On Kaggle: add it as a dataset (Add Input in the right sidebar) "
        "or copy it to /kaggle/working/.\n"
        "  Locally:   run Notebook 1 first.\n\n"
        f"Everything currently under /kaggle/input/:\n{kaggle_input_listing}"
    )

print(f"Loading data from: {DATA_PATH}")
fmcg = pd.read_csv(DATA_PATH, parse_dates=["date"])
print(f"Loaded {fmcg.shape[0]:,} rows")
print(f"Stores: {sorted(fmcg['store'].unique())}")
print(f"Products: {sorted(fmcg['product_code'].unique())}")
print(f"Date range: {fmcg['date'].min().date()} → {fmcg['date'].max().date()}")

# Verify per-store series length
series_len = fmcg.groupby(['store','product_code']).size()
print(f"\nRows per (store, product): min={series_len.min()}, max={series_len.max()} "
      f"(expected 1826 = 5 years × 365.2 days)")

# Per-store autocorrelation report (honest characterisation of task difficulty)
print("\nPer-store lag-7 autocorrelation (determines how hard naive is to beat):")
for p in sorted(fmcg['product_code'].unique()):
    lags = []
    for s in sorted(fmcg['store'].unique()):
        y = fmcg[(fmcg['store']==s)&(fmcg['product_code']==p)].sort_values('date')['sales']
        lags.append(float(y.corr(y.shift(7))))
    print(f"  {p}: mean r={np.mean(lags):.3f}, R²={np.mean(lags)**2*100:.1f}% "
          f"(naive explains this much variance per-store)")

fmcg.head()


## 1. Core experiment building blocks

These mirror `backend/app/ml/pipeline/cold_start.py` and
`backend/app/ml/evaluation/metrics.py` exactly, reproduced inline here so
this notebook is fully self-contained and Kaggle-runnable without needing
the backend package on the Python path.


In [ ]:
# Cell 4 — Cold-start density slicing + walk-forward folds
DENSITY_LEVELS = [5, 15, 30, 50, 75, 100]
MIN_WALK_FORWARD_FOLDS = 6

def temporal_density_slice(series, density_pct, date_col='date'):
    ordered = series.sort_values(date_col).reset_index(drop=True)
    cutoff = max(1, int(len(ordered) * density_pct / 100))
    return ordered.iloc[:cutoff].copy()

def walk_forward_folds(series, date_col='date', horizon=7, min_folds=MIN_WALK_FORWARD_FOLDS):
    ordered = series.sort_values(date_col).reset_index(drop=True)
    n = len(ordered)
    usable_for_folds = min_folds * horizon
    min_train_size = max(14, n - usable_for_folds)
    if min_train_size >= n:
        min_train_size = max(7, n // 2)
    folds = []
    cursor = min_train_size
    while cursor + horizon <= n and len(folds) < min_folds:
        folds.append((ordered.iloc[:cursor], ordered.iloc[cursor:cursor + horizon]))
        cursor += horizon
    return folds

print("Density levels:", DENSITY_LEVELS)


In [ ]:
# Cell 5 — Evaluation metrics + Diebold-Mariano test (Newey-West HAC variance)
def rmse(y_true, y_pred): return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
def mae(y_true, y_pred):  return float(np.mean(np.abs(y_true - y_pred)))

def mape(y_true, y_pred, eps=1e-8):
    denom = np.where(np.abs(y_true) < eps, eps, y_true)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100)

def smape(y_true, y_pred, eps=1e-8):
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom < eps, eps, denom)
    return float(np.mean(2 * np.abs(y_true - y_pred) / denom) * 100)

def compute_all_metrics(y_true, y_pred):
    return {"rmse": rmse(y_true, y_pred), "mae": mae(y_true, y_pred),
            "mape": mape(y_true, y_pred), "smape": smape(y_true, y_pred)}

def diebold_mariano_test(y_true, y_pred_model, y_pred_baseline, loss="squared", h=7):
    """
    Diebold-Mariano (1995) test with Newey-West HAC variance for h-step-ahead
    forecasts (Harvey, Leybourne & Newbold, 1997 correction for h > 1).
    H0: equal predictive accuracy. significant_at_05 requires BOTH p < 0.05
    AND d_bar < 0 (model is directionally better, not just different).
    """
    e_m = y_true - y_pred_model
    e_b = y_true - y_pred_baseline
    loss_m = e_m**2 if loss == "squared" else np.abs(e_m)
    loss_b = e_b**2 if loss == "squared" else np.abs(e_b)
    d = loss_m - loss_b
    n = len(d)
    d_bar = float(np.mean(d))

    # Newey-West HAC: gamma_0 + 2 * sum_{lag=1}^{h-1} (1 - lag/h) * gamma_lag
    # This is the correct variance estimator when h > 1 — the basic iid estimator
    # (np.var(d)/n) ignores serial correlation in multi-step loss differentials.
    gamma0 = float(np.var(d, ddof=0))
    hac = gamma0
    for lag in range(1, h):
        if n > lag:
            gamma_lag = float(np.mean((d[lag:] - d_bar) * (d[:-lag] - d_bar)))
            hac += 2 * (1 - lag / h) * gamma_lag
    var_d_bar = hac / n if n > 0 else np.nan

    if var_d_bar <= 0 or np.isnan(var_d_bar):
        return {"dm_statistic": 0.0, "p_value": 1.0,
                "significant_at_05": False, "mean_loss_differential": d_bar}
    dm_stat = d_bar / np.sqrt(var_d_bar)
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return {"dm_statistic": float(dm_stat), "p_value": float(p_value),
            "significant_at_05": bool(p_value < 0.05 and d_bar < 0),
            "mean_loss_differential": d_bar}

print("Metrics + Newey-West DM test defined (h=7 default).")


## 2. Model wrappers

Five model classes, exactly as specified in proposal Table 5: Naive
(baseline / H0), SARIMA, Prophet, XGBoost, N-BEATS. Each degrades
gracefully to the naive baseline if its underlying library is unavailable
or if the cold-start density is too low to fit it meaningfully — this
degradation is itself a finding worth visualizing, not a bug to hide.


In [ ]:
# Cell 6 — Naive last-week-sales baseline
class NaiveBaseline:
    def fit(self, y):
        self.history = y.reset_index(drop=True)
        return self
    def predict(self, n_periods):
        if len(self.history) == 0:
            return np.zeros(n_periods)
        if len(self.history) < 7:
            return np.full(n_periods, float(self.history.mean()))
        last_week = self.history.iloc[-7:].values
        reps = int(np.ceil(n_periods / 7))
        return np.tile(last_week, reps)[:n_periods]


In [ ]:
# Cell 7 — SARIMA wrapper (auto_arima order selection, weekly seasonality)
def fit_predict_sarima(y_train, n_periods, seasonal_period=7):
    if not _SARIMA_OK:
        return None
    seasonal = len(y_train) >= 2 * seasonal_period
    model = pm.auto_arima(y_train.values, seasonal=seasonal,
                           m=seasonal_period if seasonal else 1,
                           suppress_warnings=True, error_action="ignore",
                           stepwise=True, max_p=3, max_q=3, max_P=2, max_Q=2)
    preds = model.predict(n_periods=n_periods)
    return np.clip(np.asarray(preds), 0, None)


In [ ]:
# Cell 8 — Prophet wrapper with Rwanda holiday calendar injected
from datetime import date, timedelta

FIXED_PUBLIC_HOLIDAYS_MMDD = [(1,1),(1,2),(2,1),(4,7),(5,1),(7,1),(7,4),(8,15),(12,25),(12,26)]
HOLIDAY_NAMES = {(1,1):"New Year's Day",(1,2):"New Year's Holiday",(2,1):"National Heroes' Day",
                 (4,7):"Genocide Memorial Day",(5,1):"Labour Day",(7,1):"Independence Day",
                 (7,4):"Liberation Day",(8,15):"Assumption Day",(12,25):"Christmas Day",(12,26):"Boxing Day"}

def easter_sunday(year):
    a=year%19; b=year//100; c=year%100; d=b//4; e=b%4; f=(b+8)//25; g=(b-f+1)//3
    h=(19*a+b-d-g+15)%30; i=c//4; k=c%4; l=(32+2*e+2*i-h-k)%7; m=(a+11*h+22*l)//451
    month=(h+l-7*m+114)//31; day=((h+l-7*m+114)%31)+1
    return date(year, month, day)

def first_friday_of_august(year):
    d = date(year, 8, 1)
    while d.weekday() != 4: d += timedelta(days=1)
    return d

EID_AL_FITR_BY_YEAR = {2013:date(2013,8,8),2014:date(2014,7,28),2015:date(2015,7,17),2016:date(2016,7,6),2017:date(2017,6,25)}
EID_AL_ADHA_BY_YEAR = {2013:date(2013,10,15),2014:date(2014,10,4),2015:date(2015,9,24),2016:date(2016,9,12),2017:date(2017,9,1)}

def build_holiday_set(years):
    holidays = {}
    for year in years:
        for mmdd in FIXED_PUBLIC_HOLIDAYS_MMDD:
            holidays[date(year, *mmdd)] = HOLIDAY_NAMES[mmdd]
        easter = easter_sunday(year)
        holidays[easter - timedelta(days=2)] = "Good Friday"
        holidays[easter + timedelta(days=1)] = "Easter Monday"
        holidays[first_friday_of_august(year)] = "Umuganura Day"
        if year in EID_AL_FITR_BY_YEAR: holidays[EID_AL_FITR_BY_YEAR[year]] = "Eid al-Fitr"
        if year in EID_AL_ADHA_BY_YEAR: holidays[EID_AL_ADHA_BY_YEAR[year]] = "Eid al-Adha"
    return holidays

def fit_predict_prophet(dates_train, y_train, n_periods):
    if not _PROPHET_OK:
        return None
    years = sorted(pd.to_datetime(dates_train).dt.year.unique().tolist())
    if years: years.append(years[-1] + 1)
    holiday_set = build_holiday_set(years)
    holidays_df = pd.DataFrame({"holiday": list(holiday_set.values()), "ds": pd.to_datetime(list(holiday_set.keys()))})
    df = pd.DataFrame({"ds": pd.to_datetime(dates_train).values, "y": y_train.values})
    m = Prophet(holidays=holidays_df, weekly_seasonality=True,
                yearly_seasonality=len(df) >= 365, daily_seasonality=False)
    import logging
    logging.getLogger('prophet').setLevel(logging.WARNING)
    logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
    m.fit(df)
    future = m.make_future_dataframe(periods=n_periods, include_history=False)
    forecast = m.predict(future)
    return np.clip(forecast['yhat'].values, 0, None)


In [ ]:
# Cell 9 — XGBoost wrapper with lag features and TimeSeriesSplit grid search
# Matches the proposal ('XGBoost with grid search') and the backend implementation
# in backend/app/ml/models/xgboost_model.py.
FEATURE_COLUMNS = ["is_holiday", "is_memorial", "days_to_next_holiday", "season_flag",
                    "rain_intensity", "day_of_week", "week_of_year", "month",
                    "lag_7d", "lag_14d", "lag_28d"]

XGB_PARAM_GRID = {
    "n_estimators": [100, 200],
    "max_depth":    [3, 4, 6],
    "learning_rate": [0.05, 0.1],
}
XGB_MIN_OBS_FOR_GRID = 60  # need enough rows for 3 TimeSeriesSplit folds

def add_lag_features(df, target_col="sales"):
    out = df.copy()
    out["lag_7d"]  = out[target_col].shift(7)
    out["lag_14d"] = out[target_col].shift(14)
    out["lag_28d"] = out[target_col].shift(28)
    return out

def fit_predict_xgboost(train_df, future_df, target_col="sales"):
    if not _XGB_OK:
        return None
    df = add_lag_features(train_df, target_col).dropna(subset=FEATURE_COLUMNS + [target_col])
    fallback_mean = float(train_df[target_col].mean()) if len(train_df) else 0.0
    if len(df) < 10:
        return np.full(len(future_df), fallback_mean)
    X, y = df[FEATURE_COLUMNS], df[target_col]
    if _XGB_OK and len(df) >= XGB_MIN_OBS_FOR_GRID:
        base = xgb.XGBRegressor(
            subsample=0.8, colsample_bytree=0.8,
            objective="reg:squarederror", random_state=RANDOM_SEED
        )
        n_splits = min(3, max(2, len(df) // 30))
        search = GridSearchCV(
            base, XGB_PARAM_GRID,
            cv=TimeSeriesSplit(n_splits=n_splits),
            scoring="neg_root_mean_squared_error", n_jobs=1,
        )
        search.fit(X, y)
        model = search.best_estimator_
    else:
        # Too few rows for a meaningful grid search at this density level
        model = xgb.XGBRegressor(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            objective="reg:squarederror", random_state=RANDOM_SEED
        )
        model.fit(X, y)
    combined = add_lag_features(pd.concat([train_df, future_df]).reset_index(drop=True))
    X_future = combined[FEATURE_COLUMNS].fillna(0).iloc[-len(future_df):]
    return np.clip(model.predict(X_future), 0, None)


In [ ]:
# Cell 10 — N-BEATS wrapper (Nixtla neuralforecast)
# max_steps=500 on GPU gives the model enough gradient steps to converge
# (2.4 M params need at least 300-500 steps). Falls back to 100 on CPU
# to keep local validation runs feasible.
NBEATS_STEPS = 500 if ON_GPU else 100
print(f"N-BEATS max_steps: {NBEATS_STEPS} ({'GPU' if ON_GPU else 'CPU'} mode)")

def fit_predict_nbeats(dates_train, y_train, horizon, max_steps=NBEATS_STEPS):
    fallback_mean = float(y_train.mean()) if len(y_train) else 0.0
    min_required = horizon * 2 + horizon
    if not _NBEATS_OK or len(y_train) < min_required:
        return np.full(horizon, fallback_mean)
    input_size = min(2 * horizon, len(y_train) - horizon)
    df = pd.DataFrame({"unique_id": "series",
                       "ds": pd.to_datetime(dates_train).values,
                       "y": y_train.values})
    accel = "gpu" if ON_GPU else "cpu"
    model = NBEATS(h=horizon, input_size=input_size, max_steps=max_steps,
                   accelerator=accel, random_seed=RANDOM_SEED)
    nf = NeuralForecast(models=[model], freq="D")
    nf.fit(df=df)
    forecast = nf.predict()
    return np.clip(forecast["NBEATS"].values[:horizon], 0, None)


## 3. Run the full experiment matrix

**5 products × 6 density levels × (up to 6 walk-forward folds) × 5 model
classes.** On a CPU-only environment, SARIMA's `auto_arima` order search is
the dominant cost; expect the full matrix to take meaningfully longer on a
full Kaggle CPU session than the few seconds per fold seen in isolated
testing.

**Resumable, per-product cells.** Cell 11 defines the runner and writes
every `(product, store, density)` result to a checkpoint CSV
(`ml_experiments/results/ml_benchmark_results_raw.csv`) the moment it is
computed. The matrix itself then runs in five separate cells below — one
per product — so a multi-hour run can be split across sessions (e.g. FLOUR
today, OIL tomorrow) without losing progress on a crash: re-running a
product's cell skips whatever it already checkpointed and only computes
what's missing. For a faster smoke test, set `RUN_FULL_EXPERIMENT = False`
in Cell 11 and run only the SUGAR cell.


In [ ]:
# Cell 11 — Setup + resumable runner for the individual-store (Duka-proxy) experiment
#
# Each (store, product) pair is one series — the correct unit of analysis
# for a system that forecasts for individual shopkeepers. Metrics are
# computed per (store, product, density) and appended to a checkpoint CSV
# the moment they're computed, so a crash mid-run only costs the one
# (product, store, density) combo that was in flight.
#
# The matrix is split into one cell per product below so a multi-hour run
# can be spread across sessions. Re-running any product cell resumes:
# combos already in the checkpoint file are skipped automatically.

RUN_FULL_EXPERIMENT = True
RUN_ALL_STORES      = True   # set False → uses only stores 1-5 (~half the runtime)

ALL_STORES       = sorted(fmcg['store'].unique())
STORES_TO_RUN    = ALL_STORES if RUN_ALL_STORES else ALL_STORES[:5]
ALL_PRODUCTS     = sorted(fmcg['product_code'].unique())
DENSITIES_TO_RUN = DENSITY_LEVELS if RUN_FULL_EXPERIMENT else [30, 100]
HORIZON          = 7
MODEL_NAMES      = ['naive', 'sarima', 'prophet', 'xgboost', 'nbeats']
DM_MODEL_NAMES   = ['sarima', 'prophet', 'xgboost', 'nbeats']

# Resolve relative to the repo root regardless of whether this notebook's
# CWD is the repo root or ml_experiments/notebooks/ (the default when
# opened directly in local Jupyter/VS Code) -- avoids silently creating a
# nested ml_experiments/notebooks/ml_experiments/results/ directory.
_REPO_ROOT = next(
    (p for p in (Path('.'), Path('..'), Path('../..'))
     if (p / 'ml_experiments').is_dir() and (p / 'backend').is_dir()),
    Path('.'),
)
OUTPUT_DIR = _REPO_ROOT / "ml_experiments" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "ml_benchmark_results_raw.csv"


def _checkpointed_combos():
    """(product, store, density) triples already written to the checkpoint file."""
    if not CHECKPOINT_PATH.exists():
        return set()
    # NOTE: usecols does NOT reorder columns to match the list passed in — it
    # preserves the file's own column order. Select explicitly afterwards so
    # the tuple shape always matches (product, store, density_pct).
    done = pd.read_csv(CHECKPOINT_PATH, usecols=['product', 'store', 'density_pct'])
    done = done[['product', 'store', 'density_pct']]
    return set(done.itertuples(index=False, name=None))


def _append_checkpoint(rows):
    header = not CHECKPOINT_PATH.exists()
    pd.DataFrame(rows).to_csv(CHECKPOINT_PATH, mode='a', header=header, index=False)


def run_product(product, stores=None, densities=None, horizon=None):
    """
    Run the full store x density x fold matrix for a single product,
    writing each (store, density) result to CHECKPOINT_PATH as soon as
    it's computed. Safe to interrupt and re-run: already-checkpointed
    combos are skipped.
    """
    stores    = STORES_TO_RUN if stores is None else stores
    densities = DENSITIES_TO_RUN if densities is None else densities
    horizon   = HORIZON if horizon is None else horizon
    done      = _checkpointed_combos()
    n_skipped = 0

    for store in stores:
        store_series = (
            fmcg[(fmcg['product_code'] == product) & (fmcg['store'] == store)]
            .sort_values('date')
            .reset_index(drop=True)
        )

        for density in densities:
            if (product, store, density) in done:
                n_skipped += 1
                continue

            sliced = temporal_density_slice(store_series, density)
            folds  = walk_forward_folds(sliced, horizon=horizon)
            if not folds:
                continue

            fold_metrics = {m: [] for m in MODEL_NAMES}
            context = f"{product}/store {store}/density {density}%"

            # DM significance is computed ONCE per (store, density, model) on
            # residuals pooled across all folds, not once per individual
            # 7-day fold. A single fold gives the Newey-West HAC variance
            # estimator only n=7 points to fit h-1=6 lag terms, which
            # collapses to a degenerate (<=0) variance estimate most of the
            # time and forces a false "not significant" result regardless of
            # true model quality (verified empirically: a model 3x more
            # accurate than naive on every fold still failed to reach
            # significance ~60% of the time under the per-fold design).
            # Pooling first (n ~= 6*7 = 42) keeps the same h=7 lag structure
            # -- still correct for 7-step-ahead forecasts -- but gives the
            # estimator enough data to actually work; a null-case check
            # confirms the pooled test's false-positive rate lands right at
            # the nominal 5% level, i.e. it's properly calibrated.
            pooled_y_test = []
            pooled_naive_preds = []
            pooled_model_preds = {m: [] for m in DM_MODEL_NAMES}

            for fold_idx, (train, test) in enumerate(folds):
                y_train, y_test = train['sales'], test['sales'].values
                pooled_y_test.append(y_test)

                naive_preds = NaiveBaseline().fit(y_train).predict(len(test))
                fold_metrics['naive'].append(compute_all_metrics(y_test, naive_preds))
                pooled_naive_preds.append(naive_preds)

                try:
                    sarima_preds = fit_predict_sarima(y_train, len(test))
                    if sarima_preds is None: sarima_preds = naive_preds
                except Exception as exc:
                    print(f"  WARNING [{context}] fold {fold_idx}: sarima failed ({exc}), falling back to naive")
                    sarima_preds = naive_preds
                fold_metrics['sarima'].append(compute_all_metrics(y_test, sarima_preds))
                pooled_model_preds['sarima'].append(sarima_preds)

                try:
                    prophet_preds = fit_predict_prophet(train['date'], y_train, len(test))
                    if prophet_preds is None: prophet_preds = naive_preds
                except Exception as exc:
                    print(f"  WARNING [{context}] fold {fold_idx}: prophet failed ({exc}), falling back to naive")
                    prophet_preds = naive_preds
                fold_metrics['prophet'].append(compute_all_metrics(y_test, prophet_preds))
                pooled_model_preds['prophet'].append(prophet_preds)

                try:
                    xgb_preds = fit_predict_xgboost(train, test)
                    if xgb_preds is None: xgb_preds = naive_preds
                except Exception as exc:
                    print(f"  WARNING [{context}] fold {fold_idx}: xgboost failed ({exc}), falling back to naive")
                    xgb_preds = naive_preds
                fold_metrics['xgboost'].append(compute_all_metrics(y_test, xgb_preds))
                pooled_model_preds['xgboost'].append(xgb_preds)

                try:
                    nbeats_preds = fit_predict_nbeats(train['date'], y_train, len(test))
                except Exception as exc:
                    print(f"  WARNING [{context}] fold {fold_idx}: nbeats failed ({exc}), falling back to naive")
                    nbeats_preds = naive_preds
                fold_metrics['nbeats'].append(compute_all_metrics(y_test, nbeats_preds))
                pooled_model_preds['nbeats'].append(nbeats_preds)

            pooled_y_test_arr = np.concatenate(pooled_y_test)
            pooled_naive_arr = np.concatenate(pooled_naive_preds)
            fold_dm = {
                model_name: diebold_mariano_test(
                    pooled_y_test_arr, np.concatenate(preds), pooled_naive_arr, h=HORIZON
                )
                for model_name, preds in pooled_model_preds.items()
            }

            new_rows = []
            for model_name in MODEL_NAMES:
                agg = {k: float(np.mean([fm[k] for fm in fold_metrics[model_name]]))
                       for k in fold_metrics[model_name][0].keys()}
                row = {'store': store, 'product': product,
                       'density_pct': density, 'model': model_name,
                       'n_observations': len(sliced),
                       'folds_evaluated': len(folds), **agg}
                if model_name in fold_dm:
                    # Field names kept as "fraction_folds_significant" /
                    # "mean_p_value" for compatibility with existing CSV
                    # columns and plotting cells below, even though each is
                    # now a single pooled result (0.0/1.0 and one p-value)
                    # rather than an average over several per-fold tests.
                    row['fraction_folds_significant'] = float(fold_dm[model_name]['significant_at_05'])
                    row['mean_p_value'] = float(fold_dm[model_name]['p_value'])
                new_rows.append(row)

            _append_checkpoint(new_rows)
            done.add((product, store, density))

        print(f"[{product} / store {store}] done")

    print(f"[{product}] finished — {n_skipped} combo(s) already checkpointed and skipped")


print(f"Experiment matrix: {len(STORES_TO_RUN)} stores × "
      f"{len(ALL_PRODUCTS if RUN_FULL_EXPERIMENT else ['SUGAR'])} products × "
      f"{len(DENSITIES_TO_RUN)} densities per product")
print(f"Checkpoint file: {CHECKPOINT_PATH} "
      f"({'exists, will resume from it' if CHECKPOINT_PATH.exists() else 'new run'})")


In [ ]:
# Cell 11a — Run FLOUR (1 of 5 products). Independently re-runnable; resumes if interrupted.
# Skip this cell during a smoke test (RUN_FULL_EXPERIMENT = False) — run only the SUGAR cell below.
run_product('FLOUR')


In [ ]:
# Cell 11b — Run OIL (2 of 5 products). Independently re-runnable; resumes if interrupted.
# Skip this cell during a smoke test (RUN_FULL_EXPERIMENT = False) — run only the SUGAR cell below.
run_product('OIL')


In [ ]:
# Cell 11c — Run RICE (3 of 5 products). Independently re-runnable; resumes if interrupted.
# Skip this cell during a smoke test (RUN_FULL_EXPERIMENT = False) — run only the SUGAR cell below.
run_product('RICE')


In [ ]:
# Cell 11d — Run SOAP (4 of 5 products). Independently re-runnable; resumes if interrupted.
# Skip this cell during a smoke test (RUN_FULL_EXPERIMENT = False) — run only the SUGAR cell below.
run_product('SOAP')


In [ ]:
# Cell 11e — Run SUGAR (5 of 5 products). Independently re-runnable; resumes if interrupted.
# During a smoke test (RUN_FULL_EXPERIMENT = False in Cell 11 above), this is the
# only product cell you need to run.
run_product('SUGAR')


In [ ]:
# Cell 11f — Collect all checkpointed results into results_df / results_agg
#
# Run this only after all five product cells above have completed (or been
# resumed to completion). It reads the checkpoint file straight off disk,
# so it also works if the products were run across separate sessions.

results_df = pd.read_csv(CHECKPOINT_PATH)

products_expected = ALL_PRODUCTS if RUN_FULL_EXPERIMENT else ['SUGAR']
missing = set(products_expected) - set(results_df['product'].unique())
if missing:
    print(f"WARNING: no checkpointed results yet for {sorted(missing)} — "
          f"run their cells above before continuing to the visualisations below.")

# Aggregate across stores for visualisation
agg_cols = {'n_observations': 'first', 'folds_evaluated': 'first',
            'rmse': 'mean', 'mae': 'mean', 'mape': 'mean', 'smape': 'mean',
            'fraction_folds_significant': 'mean', 'mean_p_value': 'mean'}
results_agg = (
    results_df.groupby(['product', 'density_pct', 'model'])
    .agg(agg_cols)
    .reset_index()
)

print(f"\nLoaded checkpointed results.")
print(f"Raw rows (per store): {len(results_df)}")
print(f"Aggregated rows (averaged across stores): {len(results_agg)}")
results_agg.head(10)


## 4. Persist raw results


In [ ]:
# Cell 12 — Save results to disk for Notebook 4 (Results Dashboard)
# Resolve relative to the repo root regardless of whether this notebook's
# CWD is the repo root or ml_experiments/notebooks/ (the default when
# opened directly in local Jupyter/VS Code) -- avoids silently creating a
# nested ml_experiments/notebooks/ml_experiments/results/ directory.
_REPO_ROOT = next(
    (p for p in (Path('.'), Path('..'), Path('../..'))
     if (p / 'ml_experiments').is_dir() and (p / 'backend').is_dir()),
    Path('.'),
)
OUTPUT_DIR = _REPO_ROOT / "ml_experiments" / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Raw per-(store, product, density) rows
raw_path = OUTPUT_DIR / "ml_benchmark_results_raw.csv"
results_df.to_csv(raw_path, index=False)
print(f"Raw results saved: {raw_path} ({len(results_df)} rows)")

# Store-averaged rows — what Notebook 4 and the thesis tables use
agg_path = OUTPUT_DIR / "ml_benchmark_results.csv"
results_agg.to_csv(agg_path, index=False)
print(f"Aggregated results saved: {agg_path} ({len(results_agg)} rows)")

# ── Copy to Google Drive (Colab only) ─────────────────────────────────────
if IN_COLAB and DRIVE_ROOT:
    import shutil
    shutil.copy(str(raw_path), f"{DRIVE_ROOT}/ml_benchmark_results_raw.csv")
    shutil.copy(str(agg_path), f"{DRIVE_ROOT}/ml_benchmark_results.csv")
    print(f"\nBoth files also saved to Google Drive: {DRIVE_ROOT}/")
    print("Download ml_benchmark_results.csv for Notebook 4.")


## 5. Visualizations — 10+ figures


In [ ]:
# Cell 13 — Figure 1: RMSE by model, faceted by product (across all density levels, mean)
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()
for ax, product in zip(axes, sorted(results_agg['product'].unique())):
    subset = results_agg[results_agg['product'] == product]
    sns.barplot(data=subset, x='model', y='rmse', hue='model', legend=False, ax=ax, palette='Set2')
    ax.set_title(product, fontweight='bold')
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=30)
for ax in axes[len(results_agg['product'].unique()):]:
    ax.axis('off')
plt.suptitle("Mean RMSE by model, per product (averaged across all density levels)", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 14 — Figure 2: RMSE vs. density level, one line per model (averaged across products)
fig, ax = plt.subplots(figsize=(11, 6))
avg_by_density_model = results_agg.groupby(['density_pct', 'model'])['rmse'].mean().reset_index()
for model_name, group in avg_by_density_model.groupby('model'):
    ax.plot(group['density_pct'], group['rmse'], marker='o', label=model_name, linewidth=2)
ax.set_title("Mean RMSE vs. cold-start data density, by model class\n(averaged across all 5 FMCG products)", fontweight='bold')
ax.set_xlabel("Data density (%)")
ax.set_ylabel("Mean RMSE")
ax.legend(title="Model")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 15 — Figure 3: sMAPE vs. density level, one line per model
fig, ax = plt.subplots(figsize=(11, 6))
avg_smape = results_agg.groupby(['density_pct', 'model'])['smape'].mean().reset_index()
for model_name, group in avg_smape.groupby('model'):
    ax.plot(group['density_pct'], group['smape'], marker='s', label=model_name, linewidth=2)
ax.set_title("Mean sMAPE vs. cold-start data density, by model class", fontweight='bold')
ax.set_xlabel("Data density (%)")
ax.set_ylabel("Mean sMAPE (%)")
ax.legend(title="Model")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 16 — Figure 4: THE THRESHOLD CURVE — fraction of folds reaching DM significance, by model and density
dm_agg = results_agg.dropna(subset=['fraction_folds_significant'])
fig, ax = plt.subplots(figsize=(12, 6))
for model_name, group in dm_agg.groupby('model'):
    avg = group.groupby('density_pct')['fraction_folds_significant'].mean().reset_index()
    ax.plot(avg['density_pct'], avg['fraction_folds_significant'], marker='o', linewidth=2.5, label=model_name)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50% of folds significant')
ax.set_title("THRESHOLD CURVE: fraction of walk-forward folds where the model\n"
             "beats the naive baseline with Diebold-Mariano significance (p<0.05)", fontweight='bold')
ax.set_xlabel("Data density (%)")
ax.set_ylabel("Fraction of folds significant")
ax.set_ylim(-0.05, 1.05)
ax.legend(title="Model")
plt.tight_layout()
plt.show()

print("This is the headline empirical finding of the capstone: the density level")
print("at which each curve first crosses a meaningful significance threshold IS")
print("the answer to the primary research question.")


In [ ]:
# Cell 17 — Figure 5: Mean DM p-value vs. density, by model (lower = more significant)
fig, ax = plt.subplots(figsize=(11, 6))
avg_p = dm_agg.groupby(['density_pct', 'model'])['mean_p_value'].mean().reset_index()
for model_name, group in avg_p.groupby('model'):
    ax.plot(group['density_pct'], group['mean_p_value'], marker='d', linewidth=2, label=model_name)
ax.axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='p = 0.05 significance threshold')
ax.set_title("Mean Diebold-Mariano p-value vs. data density, by model", fontweight='bold')
ax.set_xlabel("Data density (%)")
ax.set_ylabel("Mean p-value (lower = more significant)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cell 18 — Figure 6: Per-product threshold heatmap (model x density, color = fraction significant)
pivot_data = []
for product in sorted(results_agg['product'].unique()):
    sub = dm_agg[dm_agg['product'] == product]
    pivot = sub.pivot_table(index='model', columns='density_pct', values='fraction_folds_significant')
    pivot_data.append((product, pivot))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for ax, (product, pivot) in zip(axes, pivot_data):
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax, cbar=True)
    ax.set_title(product, fontweight='bold')
    ax.set_xlabel("Density (%)")
    ax.set_ylabel("")
for ax in axes[len(pivot_data):]:
    ax.axis('off')
plt.suptitle("Fraction of folds reaching DM significance, by product / model / density", fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 19 — Figure 7: MAE comparison, all models, at minimum (5%) vs. maximum (100%) density
extremes = results_agg[results_agg['density_pct'].isin([5, 100])]
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=extremes, x='model', y='mae', hue='density_pct', ax=ax, palette=['#c62828', '#1a6e3c'])
ax.set_title("MAE at minimum (5%) vs. maximum (100%) data density, by model", fontweight='bold')
ax.set_xlabel("")
ax.set_ylabel("Mean Absolute Error")
ax.legend(title="Density (%)")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 20 — Figure 8: Model ranking stability across density levels (rank plot)
rank_df = results_agg.copy()
rank_df['rank'] = rank_df.groupby(['product', 'density_pct'])['rmse'].rank()
avg_rank = rank_df.groupby(['density_pct', 'model'])['rank'].mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 6))
for model_name, group in avg_rank.groupby('model'):
    ax.plot(group['density_pct'], group['rank'], marker='o', linewidth=2, label=model_name)
ax.invert_yaxis()
ax.set_title("Average RMSE rank by density level (rank 1 = best)", fontweight='bold')
ax.set_xlabel("Data density (%)")
ax.set_ylabel("Average rank (lower is better)")
ax.legend(title="Model")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 21 — Figure 9: Sample size per density level (n_observations), confirming the cold-start design
fig, ax = plt.subplots(figsize=(10, 5))
n_obs = results_agg.drop_duplicates(subset=['product', 'density_pct'])[['density_pct', 'n_observations', 'product']]
sns.boxplot(data=n_obs, x='density_pct', y='n_observations', color='#1a6e3c')
ax.set_title("Training-window size (days) at each cold-start density level", fontweight='bold')
ax.set_xlabel("Data density (%)")
ax.set_ylabel("Number of observations in training window")
plt.tight_layout()
plt.show()


In [ ]:
# Cell 22 — Figure 10: Overall model leaderboard (mean RMSE across everything)
leaderboard = results_agg.groupby('model').agg(
    mean_rmse=('rmse', 'mean'), mean_mae=('mae', 'mean'), mean_smape=('smape', 'mean')
).sort_values('mean_rmse').reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(leaderboard['model'], leaderboard['mean_rmse'], color=sns.color_palette('Set2', len(leaderboard)))
ax.set_title("Overall model leaderboard: mean RMSE across all products and density levels", fontweight='bold')
ax.set_xlabel("Mean RMSE")
ax.invert_yaxis()
for bar, val in zip(bars, leaderboard['mean_rmse']):
    ax.text(val, bar.get_y() + bar.get_height()/2, f" {val:.2f}", va='center')
plt.tight_layout()
plt.show()

leaderboard


## 6. Threshold density summary table

For each (product, model) pair, the **threshold density** is the lowest
density level at which the fraction of significant folds is at least 0.5 —
i.e. the model beats the naive baseline on a majority of walk-forward folds
at that density. This table is the direct, tabular answer to the primary
research question.


In [ ]:
# Cell 23 — Compute and display the threshold density table
# Using results_agg (store-averaged) so the threshold reflects behaviour
# averaged across Duka proxies, not a single arbitrary store.
def first_threshold(group, min_fraction=0.5):
    passing = group[group['fraction_folds_significant'] >= min_fraction].sort_values('density_pct')
    if len(passing) == 0:
        return None
    return int(passing.iloc[0]['density_pct'])

dm_agg = results_agg.dropna(subset=['fraction_folds_significant'])
threshold_table = (
    dm_agg.groupby(['product', 'model'])
    .apply(first_threshold, include_groups=False)
    .reset_index(name='threshold_density_pct')
)
threshold_pivot = threshold_table.pivot(
    index='model', columns='product', values='threshold_density_pct'
)
print("Threshold density (%) at which each model first beats the naive baseline")
print("on >= 50 % of walk-forward folds, averaged across stores")
print("(NaN = never reached significance across any tested density):\n")
threshold_pivot


In [ ]:
# Cell 24 — Figure 11: Threshold density table as a heatmap
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(threshold_pivot.astype(float), annot=True, fmt='.0f', cmap='YlGn_r',
            cbar_kws={'label': 'Threshold density (%)'}, ax=ax, linewidths=0.5, linecolor='white')
ax.set_title("Minimum data density at which each model class first beats\nthe naive baseline (lower = better, faster to become useful)", fontweight='bold')
plt.tight_layout()
plt.show()


## Summary

- Ran the full 5-product × 10-store × 6-density × 5-model experiment matrix
  using walk-forward cross-validation with a 7-day horizon.
- **Evaluated at the individual-store (Duka-proxy) level** — not national
  aggregate — so every metric is representative of what a single shopkeeper
  would experience.
- Used Newey-West HAC variance estimation in the Diebold-Mariano test
  (h = 7 lags), the correct estimator for multi-step-ahead forecast
  comparison per Harvey, Leybourne & Newbold (1997).
- XGBoost used TimeSeriesSplit GridSearchCV (matching the proposal and the
  deployed backend implementation).
- N-BEATS trained for 500 steps (GPU) / 100 steps (CPU), sufficient for
  the 2.4 M parameter model to converge on per-store series.
- Produced the threshold-curve (Figure 4) and threshold-density table
  (Cell 23) — the direct empirical answers to the primary research question.
- Saved raw per-store results (`ml_benchmark_results_raw.csv`) and
  store-averaged results (`ml_benchmark_results.csv`) for Notebook 4.

**On Rwanda localisation:** The feature engineering layer is fully
implemented (14 public holidays, Genocide Memorial Day suppressor, dual
rainy-season intensity, Rwanda-specific FMCG product mapping). This layer
is designed for deployment on real Duka sales data. The underlying Kaggle
benchmark dataset has no empirical relationship to Rwandan public events,
so feature effectiveness on this proxy data cannot be claimed and is not
claimed here. This limitation is disclosed in the thesis methods section.

**Next:** Notebook 3 evaluates the NLP side — XLM-R fine-tuned on the
commerce-domain Kinyarwanda NER test set.
